# 连接 MySQL 数据库

---

**学习目标**

1. 会用 `pymysql` 连接 MySQL、执行 SQL、取回结果
2. 分清"查询"和"修改"两类操作（后者必须 `commit`）
3. 学会**参数化查询**，避免 SQL 注入
4. 知道怎么用 pandas 把查询结果直接转成 DataFrame

**为什么要学这个？**

前面读写的是本地文件。真实项目里，数据大多存在**数据库**中。
数据分析的标准流程往往是：

```
数据库 --SQL--> DataFrame --pandas--> 分析 --matplotlib/pyecharts--> 图表
```

这一课就是把最前面那一段打通。


---
## 零、先解决一个安全习惯问题

原始笔记是这么写的：

```python
conn = Connection(
    host='localhost',
    port=3306,
    user='root',
    password='123456'      # <- 密码明文写在代码里
)
```

**这样写有三个问题**：

1. **密码会跟着代码进入 Git 仓库**，仓库一旦公开，数据库等于免费送人
2. 换环境（开发/测试/生产）就得改代码，容易改错
3. 团队协作时，每个人的密码不一样，代码没法共用

**正解：把凭据放在环境变量里**（或者 `.env` 文件，并且把 `.env` 写进 `.gitignore`）。


In [ ]:
import os

# 从环境变量读取；读不到就用占位符提醒你去设置
DB_CONFIG = {
    'host': os.environ.get('DB_HOST', 'localhost'),
    'port': int(os.environ.get('DB_PORT', 3306)),
    'user': os.environ.get('DB_USER', 'root'),
    'password': os.environ.get('DB_PASSWORD', ''),      # 默认留空，绝不明文写死
    'database': os.environ.get('DB_NAME', 'book'),
}

print('当前数据库配置：')
for k, v in DB_CONFIG.items():
    # 密码不打印真实值
    print(f'  {k:10} = {"******（已隐藏）" if k == "password" and v else v!r}')


**怎么设置环境变量？**

```bash
# Windows PowerShell（当前会话有效）
$env:DB_PASSWORD = "你的密码"

# Windows CMD
set DB_PASSWORD=你的密码

# macOS / Linux
export DB_PASSWORD=你的密码
```

想长期保存，可以用 `uv` 的 `.env` 文件机制：在项目根目录建 `.env`，
写上 `DB_PASSWORD=xxx`，然后把它加进 `.gitignore`。


---
## 一、连接数据库

> **说明**：下面这段代码需要你本机**确实在运行 MySQL**。
> 如果没有，会抛出连接异常 —— 笔记里已经用 `try/except` 包起来了，
> 不会让整个 notebook 崩掉，你照样能看后面的内容。


In [ ]:
import pymysql
from pymysql import Connection, cursors

conn = None

try:
    conn = Connection(
        host=DB_CONFIG['host'],
        port=DB_CONFIG['port'],
        user=DB_CONFIG['user'],
        password=DB_CONFIG['password'],
        # 建议显式指定字符集，避免中文乱码
        charset='utf8mb4',
        # 连接超时（秒），默认可能等很久才报错
        connect_timeout=5,
    )

    # get_server_info() 返回服务器版本，能返回就说明连接成功了
    print('连接成功！MySQL 服务器版本：', conn.get_server_info())

except pymysql.err.OperationalError as e:
    print('连接失败 ->', e)
    print()
    print('排查清单：')
    print('  1. MySQL 服务起来了吗？（Windows 服务里找 MySQL 服务，看是不是"正在运行"）')
    print('  2. 端口对吗？默认是 3306')
    print('  3. 用户名密码设置了吗？（参考上面"零"那一节）')
    print('  4. 数据库 book 存在吗？可以先连上不含 database 再 CREATE DATABASE')
except Exception as e:
    print(f'其他错误：{type(e).__name__} -> {e}')


---
## 二、游标 `cursor`：真正"执行 SQL"的东西

连接对象（`conn`）只负责"接通电话"，
**真正干活的是游标（`cursor`）** —— 它相当于一个"执行器"。

| 对象 | 负责什么 |
|---|---|
| `Connection` | 建立/关闭连接、选择数据库、提交事务 |
| `Cursor` | 执行 SQL、取回结果 |


In [ ]:
if conn:
    # 切换数据库（也可以在 Connection 里直接指定 database 参数）
    conn.select_db(DB_CONFIG['database'])

    cursor = conn.cursor()
    print('游标类型：', type(cursor))

    # 查看当前有哪些表
    cursor.execute('SHOW TABLES')
    tables = cursor.fetchall()
    print('库里的表：', tables)
else:
    print('（没有建立连接，跳过。下面的代码逻辑你可以照着读）')


---
## 三、查询：`execute` + `fetch*`

`execute()` 只是"把 SQL 发给数据库"，**结果要用 `fetch` 系列方法取回来**：

| 方法 | 取回什么 |
|---|---|
| `fetchone()` | 一行（元组）；没有则返回 `None` |
| `fetchall()` | 全部行（元组的元组） |
| `fetchmany(n)` | n 行 |

**重要**：`fetch*` 取完就"消费"掉了（和迭代器一样），再取就没了。


In [ ]:
if conn:
    cursor = conn.cursor()

    try:
        # 假设库里有张表叫 tb_book
        cursor.execute('SELECT * FROM tb_book LIMIT 5')

        print('游标自己也记录了刚影响了几行：', cursor.rowcount)

        rows = cursor.fetchall()
        print(f'取回 {len(rows)} 行：')
        for row in rows:
            print('  ', row)

        # 再取一次看看（已经是空的，因为被消费完了）
        print('再取一次：', cursor.fetchall())

    except pymysql.err.ProgrammingError as e:
        print('SQL 出错了 ->', e)
        print('（可能是因为库里没有 tb_book 这张表，这很正常）')


### `fetchone()` 的实用场景：只关心一条


In [ ]:
if conn:
    cursor = conn.cursor()
    try:
        cursor.execute('SELECT COUNT(*) FROM information_schema.tables')
        total = cursor.fetchone()
        print('当前 MySQL 实例共有', total[0], '张表')
    except Exception as e:
        print('查询失败 ->', e)


---
## 四、修改数据：别忘了 `commit()`

**这是初学者最容易踩的坑。**

`INSERT` / `UPDATE` / `DELETE` 执行完之后，数据**还在"事务"里没真正落盘**。
必须调用 `conn.commit()` 才生效；不 commit 就断开连接，改动会**全部回滚**。


In [ ]:
if conn:
    cursor = conn.cursor()

    try:
        # 建一张练习用的表（如果不存在）
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS demo_student (
                id INT AUTO_INCREMENT PRIMARY KEY,
                name VARCHAR(20) NOT NULL,
                score INT
            ) DEFAULT CHARSET=utf8mb4
        ''')

        # 插入一条
        cursor.execute(
            "INSERT INTO demo_student (name, score) VALUES (%s, %s)",
            ('张三', 90),           # <- 参数化：不要用字符串拼接！
        )

        # 【关键】不 commit 的话，这次插入不会真正生效
        conn.commit()
        print('插入成功，已提交')

        # 验证
        cursor.execute('SELECT * FROM demo_student')
        print('表里现在有：', cursor.fetchall())

        # 顺手清理掉练习表，避免污染你的数据库
        cursor.execute('DROP TABLE demo_student')
        conn.commit()
        print('练习表已清理')

    except Exception as e:
        conn.rollback()             # 出错就回滚，保证数据一致性
        print(f'操作失败，已回滚：{type(e).__name__} -> {e}')
else:
    print('（没有建立连接，跳过）')


---
## 五、参数化查询：防 SQL 注入（**必学**）

### 先看错误示范

```python
name = input('请输入用户名：')
sql = f"SELECT * FROM users WHERE name = '{name}'"
cursor.execute(sql)      # 危险！
```

如果用户输入的是：

```
' OR '1'='1
```

拼出来的 SQL 就变成了：

```sql
SELECT * FROM users WHERE name = '' OR '1'='1'
```

**条件永远成立，整张表的数据都被查出来了！**
更狠的输入还能删表、改数据。

### 正确写法：用 `%s` 占位符 + 参数元组

```python
cursor.execute('SELECT * FROM users WHERE name = %s', (name,))
```

**原理**：数据库驱动会先把 SQL 语句**编译好**（确定是"一个查询语句"），
再把参数当作**纯数据**填进去。参数里就算带引号、带 `OR`，也只会被当成普通字符串。


In [ ]:
# 演示：危险写法 vs 安全写法（纯字符串层面，不连数据库）
dangerous_input = "' OR '1'='1"
user_id = 1

dangerous_sql = f"SELECT * FROM users WHERE name = '{dangerous_input}'"
print('【危险】拼出来的 SQL：')
print(' ', dangerous_sql)
print('  结果：条件变成永远为真，整表泄露\n')

# 正确写法（这里只是展示占位符的样子，真正执行需要连数据库）
safe_sql = 'SELECT * FROM users WHERE name = %s'
print('【安全】占位符写法：')
print(' ', safe_sql, ' 参数 =', (dangerous_input,))
print('  结果：整个字符串被当作"一个名字"去匹配，什么都查不到\n')

if conn:
    cursor = conn.cursor()
    try:
        cursor.execute('SELECT COUNT(*) FROM information_schema.tables WHERE table_name = %s',
                       ('不存在的表名\' OR \'1\'=\'1',))
        print('实际执行参数化查询的结果：', cursor.fetchone()[0], '（0 张表，注入失败）')
    except Exception as e:
        print('查询异常 ->', e)


> **结论**：**永远不要用 f-string 或 `+` 拼 SQL**，一律用 `%s` 占位符。
> 注意是 `%s` 而不是 `?`（`?` 是 SQLite 的写法），MySQL 驱动用 `%s`。


---
## 六、把查询结果直接读成 pandas DataFrame

数据分析时最常用的方式：**一行代码拿到 DataFrame**。


In [ ]:
import pandas as pd

if conn:
    try:
        df = pd.read_sql('SELECT * FROM information_schema.tables LIMIT 5', conn)
        print('读到的 DataFrame：')
        print(df.head())
        print('\n形状：', df.shape)
    except Exception as e:
        print('读取失败 ->', e)
else:
    print('（没有连接，这里用本地 CSV 演示 pandas 读数据的形态）')
    demo = pd.DataFrame({
        'id': [1, 2, 3],
        'name': ['张三', '李四', '王五'],
        'score': [90, 85, 78],
    })
    print(demo)
    print('\nconn 可用时，你会写得像这样：')
    print("  df = pd.read_sql('SELECT * FROM tb_book', conn)")


### 更现代的方式：SQLAlchemy 引擎

`pymysql` 要自己管连接和 commit，
用 `SQLAlchemy` 的 `create_engine` 则更省心 —— 它会帮你管理连接池，
而且 `read_sql` / `to_sql` 都能直接用。


In [ ]:
try:
    from sqlalchemy import create_engine

    # URL 格式：方言+驱动://用户:密码@主机:端口/数据库?charset=utf8mb4
    url = (
        f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
        f"?charset=utf8mb4"
    )
    # 打印时把密码打码
    print('引擎 URL（密码已打码）：', url.replace(DB_CONFIG['password'], '****') if DB_CONFIG['password'] else url)

    engine = create_engine(url, echo=False, pool_pre_ping=True)
    # pool_pre_ping=True 会在每次取连接前先探活，避免"MySQL server has gone away"

    with engine.connect() as c:
        from sqlalchemy import text
        result = c.execute(text('SELECT VERSION()')).fetchone()
        print('通过 SQLAlchemy 连上了，MySQL 版本：', result[0])
        print('注意：新版 SQLAlchemy 2.x 要求用 text() 包装原生 SQL')

except Exception as e:
    print(f'SQLAlchemy 连接失败（正常，如果本机没有 MySQL）：{type(e).__name__} -> {e}')
    print()
    print('但引擎的写法值得记住：')
    print('  df = pd.read_sql("SELECT * FROM tb_book", engine)')
    print('  df.to_sql("新表名", engine, if_exists="append", index=False)')


---
## 七、收尾：关闭连接

和文件一样，**用完要关**。用 `with` 或者显式 `close()`。


In [ ]:
if conn:
    cursor.close()
    conn.close()
    print('连接已关闭')
else:
    print('（本来就没有连接）')


---
## 八、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 密码明文写在代码里 | 安全风险 | 用环境变量 / `.env` + `.gitignore` |
| 改了数据没 `commit()` | 程序看着成功，数据库里没变 | 增删改后必须 `conn.commit()` |
| 用 f-string 拼 SQL | SQL 注入风险 | 用 `%s` 占位符 + 参数元组 |
| `fetchall()` 调了两次 | 第二次是空的 | `fetch*` 会消费结果，先存进变量 |
| 忘了指定 `charset='utf8mb4'` | 中文乱码 / emoji 存不进去 | 连接时显式指定 |
| `conn` 用完不 `close()` | 连接泄漏，最终连不上 | `with` 或显式 `close()` |
| 没有超时设置 | 连不上时卡很久 | `connect_timeout=5` |
| 直接把 `conn` 当游标用 | `AttributeError` | 要先 `conn.cursor()` |
| SQLAlchemy 1.x 写法用在 2.x | 报错要求 `text()` | 用 `text('SELECT ...')` 包装 |
| 用 `?` 当占位符 | 语法错误 | MySQL 驱动用 `%s` |
| 表名/字段名用中文没加反引号 | 语法错误 | 用反引号 `` `中文表名` `` 或用英文名 |

---

## 九、动手练习

**练习 1**：写一个 `query(sql, params=None)` 函数，返回查询结果的列表，
自动处理连接、游标和异常。要求用 `with` 保证连接一定关闭。

**练习 2**：写一个 `insert_student(name, score)` 函数，
用参数化查询插入一条记录并 commit。

**练习 3**：把 `demo_student` 表的数据读成 DataFrame，
算出平均分，用 pyecharts 画一个柱状图。

**练习 4**：写一个"从 CSV 批量导入数据库"的脚本：
读 `data/地图数据/疫情统计.csv`，用 `executemany` 批量插入。
提示：
```python
cursor.executemany('INSERT INTO t (a, b) VALUES (%s, %s)', rows)
```

**练习 5**：解释下面两段代码的区别，说明第二段为什么危险：
```python
# A
cursor.execute('SELECT * FROM t WHERE id = %s', (user_id,))
# B
cursor.execute(f'SELECT * FROM t WHERE id = {user_id}')
```

---

**答案思路（先自己做）**

```python
# 练习 1
from contextlib import closing

def query(sql, params=None):
    """执行查询并返回所有结果"""
    with closing(Connection(**DB_CONFIG)) as conn:
        with closing(conn.cursor()) as cur:
            cur.execute(sql, params)
            return cur.fetchall()


# 练习 2
def insert_student(name, score):
    conn = Connection(**DB_CONFIG)
    try:
        with conn.cursor() as cur:
            cur.execute(
                'INSERT INTO demo_student (name, score) VALUES (%s, %s)',
                (name, score),
            )
        conn.commit()
        return True
    except Exception as e:
        conn.rollback()
        print('插入失败：', e)
        return False
    finally:
        conn.close()
```
